In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
import re
import pandas as pd
from PIL import Image
import pytesseract
import pdfplumber
from pdf2image import convert_from_path
from docx import Document
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
from sqlalchemy import create_engine

POPPLER_PATH = r"C:\Program Files (x86)\poppler-25.12.0\Library\bin"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

class DataExtractor:

    def __init__(self):
        self.records = []

    def extract_text_from_image(self, path):
        return pytesseract.image_to_string(Image.open(path))

    # PDF text extraction done in both types
    def extract_text_from_pdf(self, path):
        text = ""
        try:
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    if page.extract_text():
                        text += page.extract_text() + "\n"
        except:
            pass

        if not text.strip():
            images = convert_from_path(path, poppler_path=POPPLER_PATH)
            for img in images:
                text += pytesseract.image_to_string(img)

        return text

    def extract_text_from_docx(self, path):
        doc = Document(path)
        return "\n".join(p.text for p in doc.paragraphs)
        
    def extract_from_excel(self, path):
        df = pd.read_excel(path)
        for _, row in df.iterrows():
            text = ""
            for c, v in row.items():
                if pd.notna(v):
                    text += f"{c}: {v}\n"
            self.records.append(self.parse_data(text))
        print(f"✓ Extracted: {os.path.basename(path)}")

    def extract_from_json(self, path):
        df = pd.read_json(path)
        for _, row in df.iterrows():
            text = ""
            for c, v in row.items():
                if pd.notna(v):
                    text += f"{c}: {v}\n"
            self.records.append(self.parse_data(text))
        print(f"✓ Extracted: {os.path.basename(path)}")

    def extract_from_txt(self, path):
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()

        blocks = content.split("-" * 40)
        for block in blocks:
            if block.strip():
                self.records.append(self.parse_data(block))

        print(f"✓ Extracted: {os.path.basename(path)}")

    def extract_from_html(self, path):
        with open(path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "html.parser")

        rows = soup.find_all("tr")[1:]
        for r in rows:
            cols = [c.get_text(strip=True) for c in r.find_all("td")]
            if len(cols) >= 9:
                text = f"""
                Name: {cols[1]}
                ID: {cols[2]}
                Email: {cols[3]}
                Phone: {cols[4]}
                DOB: {cols[5]}
                Gender: {cols[6]}
                Course: {cols[7]}
                CGPA: {cols[8]}
                """
                self.records.append(self.parse_data(text))

        print(f"✓ Extracted: {os.path.basename(path)}")

    def extract_from_xml(self, path):
        tree = ET.parse(path)
        root = tree.getroot()

        for student in root.findall(".//Student"):
            text = ""
            for child in student:
                if child.text:
                    text += f"{child.tag}: {child.text}\n"
            self.records.append(self.parse_data(text))

        print(f"✓ Extracted: {os.path.basename(path)}")

    def parse_data(self, text):
        patterns = {
            "Name": r"Name\s*[:\-]?\s*([A-Za-z ]+)",
            "ID": r"ID\s*[:\-]?\s*(\d+)",
            "Email": r"([\w\.-]+@[\w\.-]+)",
            "Phone": r"\b(\d{9,10})\b",
            "DOB": r"(\d{2}[-/]\d{2}[-/]\d{4})",
            "Gender": r"\b(Male|Female|Other)\b",
            "Course": r"(B\.?Tech|BCA|MCA|MBA|BBA|BA|B\.Sc|B\.Com|CSE|Degree|Inter)",
            "CGPA": r"(?:CGPA|Marks|Score|Percentage)\s*[:\-]?\s*(\d+(?:\.\d+)?)"
        }
        data = {}
        for field, pat in patterns.items():
            m = re.search(pat, text, re.I)
            value = m.group(1) if m else ""

            #phone must be string
            if field == "Phone":
                value = str(value)

            data[field] = value

        return data
    
    def process_file(self, path): # Files reading
        p = path.lower()

        if p.endswith((".jpg", ".jpeg", ".png")):
            self.records.append(self.parse_data(self.extract_text_from_image(path)))

        elif p.endswith(".pdf"):
            self.records.append(self.parse_data(self.extract_text_from_pdf(path)))

        elif p.endswith(".docx"):
            self.records.append(self.parse_data(self.extract_text_from_docx(path)))

        elif p.endswith(".xlsx"):
            self.extract_from_excel(path)
            return

        elif p.endswith(".json"):
            self.extract_from_json(path)
            return

        elif p.endswith(".txt"):
            self.extract_from_txt(path)
            return

        elif p.endswith((".html", ".htm")):
            self.extract_from_html(path)
            return

        elif p.endswith(".xml"):
            self.extract_from_xml(path)
            return

        print(f"✓ Extracted: {os.path.basename(path)}")

    def process_path(self, path):
        if os.path.isfile(path):
            self.process_file(path)
        else:
            for f in os.listdir(path):
                self.process_file(os.path.join(path, f))

    # to transver data into
    def to_dataframe(self):
        df = pd.DataFrame(self.records)

        if "Name" in df.columns:
           df = df.rename(columns={"Name": "FULLNAME"})

        return df
    
    def save_to_csv(self, path):
        df = self.to_dataframe()
        df.to_csv(path, index=False)
        print(f"✓ CSV Created: {path}")
        print(f"✓ Records Saved: {len(df)}")

In [2]:
# EXECUTION OF CODE
extractor = DataExtractor()
extractor.process_path(r"C:\Users\prane\Downloads\praneeth_project\data_forms")
df = extractor.to_dataframe()
print(df)

csv_path = r"C:\Users\prane\Downloads\praneeth_project\all_data_final_output.csv"
extractor.save_to_csv(csv_path)

✓ Extracted: 1data_jpg.jpg
✓ Extracted: 2data_pdf.pdf
✓ Extracted: 3data_doc.docx
✓ Extracted: 4data_xlsx.xlsx
✓ Extracted: 5data_json.json
✓ Extracted: 6data_txt.txt
✓ Extracted: 7data_html.html
✓ Extracted: 8data_xml.xml
              FULLNAME      ID                    Email       Phone  \
0              Kavitha    1007   kavithareddy@gmail.com  9012345678   
1         Rohit Sharma    2003   rohit.sharma@gmail.com  9876543203   
2          Sanya Reddy    2002    sanya.reddy@gmail.com  9876543202   
3       Samar Majumdar  100001  student100001@gmail.com  9642995839   
4      Uthkarsh Bakshi  100002  student100002@gmail.com  2046202955   
...                ...     ...                      ...         ...   
49998      Taran Ravel   99996   student99996@gmail.com  4038267713   
49999      Shaan Karan   99997   student99997@gmail.com  8863633611   
50000   Dishani Chahal   99998   student99998@gmail.com  7404692015   
50001        Emir Dhar   99999   student99999@gmail.com  7643112408

In [3]:
print(df.columns)

Index(['FULLNAME', 'ID', 'Email', 'Phone', 'DOB', 'Gender', 'Course', 'CGPA'], dtype='object')


In [ ]:
# MYSQL DATABASE LOAD
engine = create_engine("mysql+pymysql://root:TG@127.0.0.1:3306/unv_extractor")

df = pd.read_csv(csv_path)
print("✓ CSV Loaded")
print(df.head(5))
print(df.tail(5))

df.to_sql(name="all_data",con=engine,index=False,if_exists="replace")

engine.dispose()
print("✓ Data inserted into MySQL")
print("✓ MySQL connection closed")
print("Total rows:", len(df))

✓ CSV Loaded
          FULLNAME      ID                    Email         Phone         DOB  \
0          Kavitha    1007   kavithareddy@gmail.com  9.012346e+09  03/11/2004   
1     Rohit Sharma    2003   rohit.sharma@gmail.com  9.876543e+09  09/11/2002   
2      Sanya Reddy    2002    sanya.reddy@gmail.com  9.876543e+09  21/08/2004   
3   Samar Majumdar  100001  student100001@gmail.com  9.642996e+09  08/01/2004   
4  Uthkarsh Bakshi  100002  student100002@gmail.com  2.046203e+09  08/09/2000   

   Gender  Course  CGPA  
0  Female  B.Tech  9.90  
1    Male     MBA  8.90  
2  Female     BCA  7.60  
3  Female     BCA  8.83  
4    Male      Ba  6.64  
              FULLNAME      ID                    Email         Phone  \
49998      Taran Ravel   99996   student99996@gmail.com  4.038268e+09   
49999      Shaan Karan   99997   student99997@gmail.com  8.863634e+09   
50000   Dishani Chahal   99998   student99998@gmail.com  7.404692e+09   
50001        Emir Dhar   99999   student99999@gmail.